<a href="https://colab.research.google.com/github/YeseniaL2025/Fundamentos-Python/blob/main/Copia_de_proyecto_final_VERSION_5_tva.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#paso 1
#adjunto mi zip con las imagnes
from google.colab import files
uploaded = files.upload()

KeyboardInterrupt: 

In [ ]:
#paso 2
#descomprime el zip
import os, zipfile

zip_path = list(uploaded.keys())[0]  # obtiene el nombre del archivo subido
extract_dir = './data'
os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print("Archivos descomprimidos en:", extract_dir)


In [ ]:
#paso 3
#verifico las imagenes
import os

exts = ('.jpg', '.jpeg', '.png')
total = 0
for root, dirs, files in os.walk(extract_dir):
    imgs = [f for f in files if f.lower().endswith(exts)]
    if imgs:
        print(f"{root}: {len(imgs)} imágenes")
        total += len(imgs
)
print("Total de imágenes:", total)


In [ ]:
# paso 4
# Creamos un DataFrame con etiquetas
import pandas as pd
import re
import os

# --- Rangos de edad oficiales ---
# (0-2), (4-6), (8-12), (15-20), (25-32), (38-43), (48-53), (60-100)
def asignar_rango(edad):
    """
    Devuelve la clase (0..7) según los rangos oficiales:
      0: 0-2
      1: 4-6
      2: 8-12
      3: 15-20
      4: 25-32
      5: 38-43
      6: 48-53
      7: 60-100
    Si la edad no cae en ninguno de estos rangos, retorna None.
    """
    if 0 <= edad <= 2:
        return 0
    elif 4 <= edad <= 6:
        return 1
    elif 8 <= edad <= 12:
        return 2
    elif 15 <= edad <= 20:
        return 3
    elif 25 <= edad <= 32:
        return 4
    elif 38 <= edad <= 43:
        return 5
    elif 48 <= edad <= 53:
        return 6
    elif 60 <= edad <= 100:
        return 7
    else:
        return None

# Extensiones de imagen válidas
exts = ('.jpg', '.jpeg', '.png')

# Patrón: "edad-genero.ext" donde genero es 0 (Masculino) o 1 (Femenino)
# Ejemplos válidos: 2-0.jpg, 18-1.jpeg, 65-0.png
#pat = re.compile(r'^(\d+)-([01])\.(jpg|jpeg|png)$', re.IGNORECASE)
pat = re.compile(r'^(\d+)-([01]{1,2})\.(jpg|jpeg|png)$', re.IGNORECASE)
#pat = re.compile(r'^(\d+)-([01]{1,2})\.(jpg|jpeg|png)$', re.IGNORECASE)
data = []
descartadas = 0
no_coinciden_regex = 0

# 'extract_dir' debe existir del PASO 2
for root, dirs, files in os.walk(extract_dir):
    for fname in files:
        if fname.lower().endswith(exts):
            m = pat.match(fname)
            if not m:
                no_coinciden_regex += 1
                continue
            edad = int(m.group(1))
            genero = int(m.group(2))  # 0=Masculino, 1=Femenino

            if genero not in [0,1]:
              continue  # descarta si no es válido

            rango = asignar_rango(edad)
            if rango is None:
                descartadas += 1
                continue
            data.append([os.path.join(root, fname), rango, genero])

# Construcción del DataFrame
df = pd.DataFrame(data, columns=['filename', 'age_class', 'gender'])

# Resumen informativo
print("Ejemplo de filas:")
print(df.head(8))
print(f"Total de imágenes etiquetadas: {len(df)}")
print(f"Imágenes descartadas por edad fuera de rango: {descartadas}")
print(f"Archivos de imagen que no coincidieron con el patrón 'edad-genero.ext': {no_coinciden_regex}")

# Conteos por clase de edad
print("\nDistribución por clase de edad (0..7):")
print(df['age_class'].value_counts().sort_index())

# Conteos por género
print("\nDistribución por género (0=Masculino, 1=Femenino):")
print(df['gender'].value_counts().sort_index())


In [ ]:
#complemento paso 4 muestra la cantidad de imagenes que tenemos por edad y genero

import matplotlib.pyplot as plt

# Distribución por clase de edad
plt.figure(figsize=(8,5))
df['age_class'].value_counts().sort_index().plot(kind='bar', color='skyblue')
plt.title("Distribución por rango de edad")
plt.xlabel("Clase de edad (0..7)")
plt.ylabel("Número de imágenes")
plt.xticks(rotation=0)
plt.show()

# Distribución por género
plt.figure(figsize=(6,4))
df['gender'].value_counts().sort_index().plot(kind='bar', color=['lightcoral','lightblue'])
plt.title("Distribución por género")
plt.xlabel("Género (0=Hombre, 1=Mujer)")
plt.ylabel("Número de imágenes")
plt.xticks(rotation=0)
plt.show()

In [ ]:
#paso 5
#dividimos en entreamiento y validacion

from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df[['age_class','gender']])

In [ ]:
!pip install --upgrade tensorflow keras

In [ ]:
#paso 6
#Crear datasets con tf.data

import tensorflow as tf

# Configuración
img_size = (128, 128)
batch_size = 32
num_age_classes = 8

# Función para leer y preprocesar una imagen desde la ruta
def load_and_preprocess_image(path):
    img_bytes = tf.io.read_file(path)
    img = tf.image.decode_image(img_bytes, channels=3, expand_animations=False)
    img = tf.image.convert_image_dtype(img, tf.float32)          # [0,1]
    img = tf.image.resize(img, img_size)
    return img

# Función que arma el ejemplo: (imagen, {edad_onehot, genero})
def make_example(path, age_class, gender):
    img = load_and_preprocess_image(path)
    age_onehot = tf.one_hot(age_class, depth=num_age_classes)     # one-hot
    gender = tf.cast(gender, tf.float32)                          # 0.0 / 1.0
    return img, {'age_output': age_onehot, 'gender_output': gender}

# Extraer tensores desde los DataFrames
train_paths   = tf.constant(train_df['filename'].values)
train_ages    = tf.constant(train_df['age_class'].values, dtype=tf.int32)
train_genders = tf.constant(train_df['gender'].values,    dtype=tf.int32)

val_paths   = tf.constant(val_df['filename'].values)
val_ages    = tf.constant(val_df['age_class'].values, dtype=tf.int32)
val_genders = tf.constant(val_df['gender'].values,    dtype=tf.int32)

# Crear datasets con tf.data
train_ds = tf.data.Dataset.from_tensor_slices((train_paths, train_ages, train_genders))
train_ds = train_ds.map(make_example, num_parallel_calls=tf.data.AUTOTUNE)
train_ds = train_ds.shuffle(2048).batch(batch_size).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((val_paths, val_ages, val_genders))
val_ds = val_ds.map(make_example, num_parallel_calls=tf.data.AUTOTUNE)
val_ds = val_ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)



In [ ]:
#paso 7
#Definir el modelo CNN y entrenar con data augmentation

import tensorflow as tf
import numpy as np
from tensorflow.keras import Model, Sequential
from tensorflow.keras.layers import (Input, Conv2D, MaxPooling2D, Flatten,
                                     Dense, Dropout, RandomFlip, RandomRotation, RandomZoom)

num_age_classes = 8

# Bloque de augmentación (solo se aplica en entrenamiento)
data_augmentation = Sequential([
    RandomFlip('horizontal'),
    RandomRotation(0.05),
    RandomZoom(0.1)
], name='augmentation')

inputs = Input(shape=(img_size[0], img_size[1], 3))
x = data_augmentation(inputs)

x = Conv2D(32, (3,3), activation='relu')(x)
x = MaxPooling2D((2,2))(x)
x = Conv2D(64, (3,3), activation='relu')(x)
x = MaxPooling2D((2,2))(x)
x = Conv2D(128, (3,3), activation='relu')(x)
x = MaxPooling2D((2,2))(x)
x = Flatten()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)

age_out = Dense(num_age_classes, activation='softmax', name='age_output')(x)
gender_out = Dense(1, activation='sigmoid', name='gender_output')(x)

model = Model(inputs=inputs, outputs=[age_out, gender_out])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss={'age_output': 'categorical_crossentropy', 'gender_output': 'binary_crossentropy'},
    metrics={'age_output': 'accuracy', 'gender_output': 'accuracy'}
)

model.summary()

# Entrenar
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

# --- Métricas adicionales ---
from sklearn.metrics import classification_report

val_images, val_age_labels, val_gender_labels = [], [], []
for batch in val_ds:
    imgs, labels = batch
    val_images.append(imgs.numpy())
    val_age_labels.append(labels['age_output'].numpy())
    val_gender_labels.append(labels['gender_output'].numpy())

val_images = np.concatenate(val_images)
val_age_labels = np.argmax(np.concatenate(val_age_labels), axis=1)
val_gender_labels = np.concatenate(val_gender_labels)

age_preds, gender_preds = model.predict(val_images)
age_preds_classes = np.argmax(age_preds, axis=1)
gender_preds_classes = (gender_preds > 0.5).astype(int).flatten()

print("Reporte de clasificación para Edad:")
print(classification_report(val_age_labels, age_preds_classes))

print("Reporte de clasificación para Género:")
print(classification_report(val_gender_labels, gender_preds_classes))


In [ ]:
# Gráficas para ver como evoluciona el modelo de accuracy y loss
plt.figure(figsize=(12,5))

# Accuracy
plt.subplot(1,2,1)
plt.plot(history.history['age_output_accuracy'], label='Edad - Entrenamiento')
plt.plot(history.history['val_age_output_accuracy'], label='Edad - Validación')
plt.plot(history.history['gender_output_accuracy'], label='Género - Entrenamiento')
plt.plot(history.history['val_gender_output_accuracy'], label='Género - Validación')
plt.title("Accuracy durante el entrenamiento")
plt.xlabel("Épocas")
plt.ylabel("Accuracy")
plt.legend()

# Loss
plt.subplot(1,2,2)
plt.plot(history.history['age_output_loss'], label='Edad - Entrenamiento')
plt.plot(history.history['val_age_output_loss'], label='Edad - Validación')
plt.plot(history.history['gender_output_loss'], label='Género - Entrenamiento')
plt.plot(history.history['val_gender_output_loss'], label='Género - Validación')
plt.title("Loss durante el entrenamiento")
plt.xlabel("Épocas")
plt.ylabel("Loss")
plt.legend()

plt.show()

In [ ]:
# --- Visualización de predicciones ---
import matplotlib.pyplot as plt

# Seleccionar algunas imágenes de validación
sample_imgs = val_images[:10]
sample_age_true = val_age_labels[:10]
sample_gender_true = val_gender_labels[:10].astype(int)   # 👈 convertir a enteros

age_preds, gender_preds = model.predict(sample_imgs)
age_preds_classes = np.argmax(age_preds, axis=1)
gender_preds_classes = (gender_preds > 0.5).astype(int).flatten()

age_ranges = {
    0: "0-2 años", 1: "4-6 años", 2: "8-12 años", 3: "15-20 años",
    4: "25-32 años", 5: "38-43 años", 6: "48-53 años", 7: "60-100 años"
}

plt.figure(figsize=(15,8))
for i in range(len(sample_imgs)):
    plt.subplot(2,5,i+1)
    plt.imshow(sample_imgs[i])
    plt.axis("off")
    plt.title(
        f"Edad: {age_ranges[sample_age_true[i]]}/{age_ranges[age_preds_classes[i]]}\n"
        f"Género: {['M','F'][sample_gender_true[i]]}/{['M','F'][gender_preds_classes[i]]}"
    )
plt.suptitle("Ejemplos de predicciones (Real/Predicho)")
plt.show()

In [ ]:
#paso 8
#Visualizar resultados de accuracy a prueba de cambios de nombres

import matplotlib.pyplot as plt

# Ver claves disponibles
print("Claves en history.history:")
print(history.history.keys())

# Helper para obtener una curva si existe
def get_curve(key):
    return history.history.get(key, [])

plt.figure(figsize=(10,5))
plt.plot(get_curve('age_output_accuracy'),       label='Edad - Train')
plt.plot(get_curve('val_age_output_accuracy'),   label='Edad - Val')
plt.plot(get_curve('gender_output_accuracy'),    label='Género - Train')
plt.plot(get_curve('val_gender_output_accuracy'),label='Género - Val')

# Si no hubo métricas por salida, intenta las generales:
if not get_curve('age_output_accuracy') and get_curve('accuracy'):
    plt.plot(get_curve('accuracy'), label='Accuracy - Train (global)')
if not get_curve('val_age_output_accuracy') and get_curve('val_accuracy'):
    plt.plot(get_curve('val_accuracy'), label='Accuracy - Val (global)')

plt.title('Progreso de Accuracy')
plt.xlabel('Épocas')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
#PASO 9
#Explicación clara de categorical_crossentropy

print("""
¿Por qué 'categorical_crossentropy' para la edad?

• Problema multiclase: la edad se modela por rangos (8 categorías).
• La red produce una distribución de probabilidad (softmax) sobre esas 8 clases.
• La etiqueta real se representa en one-hot (ej., [0,0,1,0,0,0,0,0]).
• 'categorical_crossentropy' mide la divergencia entre la distribución real y la predicha.
• Penaliza fuertemente dar poca probabilidad a la clase correcta, forzando ajustes de pesos.
• Resultado: el modelo aprende a concentrar probabilidad donde corresponde.

Para el género (binario), 'binary_crossentropy' es la elección correcta porque solo hay
dos clases (Masculino/Femenino) y la salida es una probabilidad con 'sigmoid' (0..1).
""")


In [ ]:
!pip install tensorflow

In [ ]:
!pip install keras-tuner

In [ ]:
#paso 10 — Instalar e importar Keras Tuner
!pip install -q keras-tuner

import keras_tuner as kt
import tensorflow as tf
from tensorflow.keras import Model, Sequential
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D, Flatten,
    Dense, Dropout, RandomFlip, RandomRotation, RandomZoom, BatchNormalization
)


In [ ]:
#PASO 11
# Callback para métrica compuesta

class JointAccCallback(tf.keras.callbacks.Callback):
    """Promedia val_age_output_accuracy y val_gender_output_accuracy,
    y lo registra como 'val_joint_acc' para que Keras Tuner lo optimice."""
    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        a = logs.get('val_age_output_accuracy')
        g = logs.get('val_gender_output_accuracy')
        if a is not None and g is not None:
            logs['val_joint_acc'] = (a + g) / 2.0



In [ ]:
#paso 12
#Función builder con hiperparámetros

def build_model(hp):
    img_size = (128, 128)  # Mantener igual que tu pipeline tf.data

    # Augmentación configurable
    aug_flip = hp.Boolean('aug_flip', default=True)
    aug_rot  = hp.Float('aug_rotation', 0.0, 0.15, step=0.05, default=0.05)
    aug_zoom = hp.Float('aug_zoom',     0.0, 0.25, step=0.05, default=0.10)

    data_augmentation = Sequential(name='augmentation')
    if aug_flip: data_augmentation.add(RandomFlip('horizontal'))
    if aug_rot > 0: data_augmentation.add(RandomRotation(aug_rot))
    if aug_zoom > 0: data_augmentation.add(RandomZoom(aug_zoom))

    inputs = Input(shape=(img_size[0], img_size[1], 3))
    x = data_augmentation(inputs)

    # Bloques conv
    use_bn = hp.Boolean('use_batchnorm', default=True)
    for i, base_filters in enumerate([32, 64, 128]):
        filters = hp.Int(f'conv{i+1}_filters', min_value=base_filters//2, max_value=base_filters*2, step=16, default=base_filters)
        x = Conv2D(filters, (3,3), activation='relu', padding='same')(x)
        if use_bn:
            x = BatchNormalization()(x)
        x = MaxPooling2D((2,2))(x)

    x = Flatten()(x)

    # Capa densa y dropout
    dense_units = hp.Int('dense_units', min_value=64, max_value=512, step=64, default=128)
    x = Dense(dense_units, activation='relu')(x)
    dropout_rate = hp.Float('dropout_rate', 0.2, 0.6, step=0.1, default=0.5)
    x = Dropout(dropout_rate)(x)

    # Salidas
    num_age_classes = 8
    age_out    = Dense(num_age_classes, activation='softmax', name='age_output')(x)
    gender_out = Dense(1, activation='sigmoid', name='gender_output')(x)

    model = Model(inputs=inputs, outputs=[age_out, gender_out])

    # Optimizador y LR
    opt_name = hp.Choice('optimizer', values=['adam', 'rmsprop', 'sgd'], default='adam')
    lr = hp.Choice('learning_rate', values=[1e-4, 3e-4, 1e-3, 3e-3], default=1e-3)

    if opt_name == 'adam':
        optimizer = tf.keras.optimizers.Adam(learning_rate=lr)
    elif opt_name == 'rmsprop':
        optimizer = tf.keras.optimizers.RMSprop(learning_rate=lr)
    else:
        momentum = hp.Float('sgd_momentum', 0.0, 0.9, step=0.1, default=0.9)
        optimizer = tf.keras.optimizers.SGD(learning_rate=lr, momentum=momentum, nesterov=True)

    model.compile(
        optimizer=optimizer,
        loss={'age_output':'categorical_crossentropy', 'gender_output':'binary_crossentropy'},
        metrics={'age_output':'accuracy', 'gender_output':'accuracy'}
    )
    return model



In [ ]:
#PASO 13
#Configurar y ejecutar el tuner


# PASO 13 — Configurar y ejecutar el tuner
assert 'train_ds' in globals() and 'val_ds' in globals(), "Ejecuta PASO 6 y 7 para crear train_ds y val_ds antes del tuning."

tuner = kt.Hyperband(
    build_model,
    objective=kt.Objective('val_joint_acc', direction='max'),
    max_epochs=15,
    factor=3,
    directory='kt_dir',
    project_name='edad_genero_tuning'
)

joint_acc_cb = JointAccCallback()

early = tf.keras.callbacks.EarlyStopping(
    monitor='val_joint_acc',
    mode='max',
    patience=3,
    restore_best_weights=True
)

tuner.search(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=[joint_acc_cb, early]
)


In [ ]:

#PASO 14
#Entrenar modelo final y graficar
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
best_model = tuner.hypermodel.build(best_hps)

history_best = best_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=[joint_acc_cb, early]
)

print("Mejores hiperparámetros encontrados:")
for hp in best_hps.values.keys():
    print(f" - {hp}: {best_hps.get(hp)}")

# Guardar modelo
best_model.save('modelo_edad_genero_tuned.keras')

# Graficar métricas
import matplotlib.pyplot as plt

def get_curve(hist, key):
    return hist.history.get(key, [])

plt.figure(figsize=(10,5))
plt.plot(get_curve(history_best, 'age_output_accuracy'),         label='Edad - Train')
plt.plot(get_curve(history_best, 'val_age_output_accuracy'),     label='Edad - Val')
plt.plot(get_curve(history_best, 'gender_output_accuracy'),      label='Género - Train')
plt.plot(get_curve(history_best, 'val_gender_output_accuracy'),  label='Género - Val')
plt.plot(get_curve(history_best, 'val_joint_acc'),               label='Métrica conjunta - Val', linestyle='--')
plt.title('Progreso de métricas (mejor configuración)')
plt.xlabel('Épocas')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()
